In [ ]:
println("Loading imports")
include("../src/naml.jl")

In [ ]:
println("Setting up configurable parameters")

# Configuration parameters - modify these to change problem size
num_dimensions = 60      # Number of variables (data + parameters combined)
num_data_points = 60    # Number of data points
num_parameters = div(num_dimensions, 2)  # Parameters take up half the variables
num_data_vars = num_dimensions - num_parameters

println("Problem configuration:")
println("  Total variables: $num_dimensions")
println("  Data variables: $num_data_vars")
println("  Parameters: $num_parameters")
println("  Data points: $num_data_points")

In [ ]:
println("Initialising data with $num_data_vars variables and $num_data_points data points")
prec = 20
K = PadicField(2, prec)

# Create num_data_points data points, each with num_data_vars dimensions
# Each data point is a vector of num_data_vars p-adic numbers
data_vals = []
for j in 1:num_data_points
    point = [K(1) for _ in 1:num_data_vars]
    # Vary one coordinate per point (cycling through dimensions)
    point[((j-1) % num_data_vars) + 1] = K(2)
    push!(data_vals, point)
end
data = [(vals, 0) for vals in data_vals]

# Create polynomial ring with num_dimensions total variables
# First num_data_vars are data variables, last num_parameters are parameters
var_names = vcat(["x$i" for i in 1:num_data_vars], ["a$i" for i in 1:num_parameters])
R, vars = polynomial_ring(K, var_names)

# Extract variables
x_vars = vars[1:num_data_vars]
a_vars = vars[(num_data_vars+1):num_dimensions]

# Create num_data_vars linear polynomials: (x_i - a_i) for each dimension
# LinearPolynomial represents: coefficients[1]*var1 + ... + coefficients[num_dimensions]*var_num_dimensions + constant
linear_polys = LinearPolynomial{typeof(K(1))}[]
for i in 1:num_data_vars
    # Coefficient for x_i is 1, coefficient for a_i is -1, all others are 0
    coeffs = zeros(K, num_dimensions)
    coeffs[i] = K(1)                  # coefficient for x_i
    coeffs[i+num_data_vars] = -K(1)   # coefficient for a_i
    push!(linear_polys, LinearPolynomial{typeof(K(1))}(coeffs, K(0)))
end

# Create the linear absolute polynomial sum
g = LinearAbsolutePolynomialSum(linear_polys)

# param_info: first num_data_vars are data variables (true), last num_parameters are parameters (false)
param_info = vcat(fill(true, num_data_vars), fill(false, num_parameters))

# Specify which variables are data and which are parameters
f = AbstractModel(g, param_info)

println("Linear model created with $(length(linear_polys)) linear polynomials")

In [ ]:
println("Initialising optimisation tools")

next_branch = 1
config = (false, 1)
# Initial parameters: num_parameters-dimensional point with center at [1,1,...,1] and radius [0,0,...,0]
initial = ValuationPolydisc(fill(K(1), num_parameters), fill(0, num_parameters))

# Create loss function using the vector-valued data directly
ℓ = MPE_loss_init(f, data, 1)

# We optimise using Greedy descent
greedy_optim::OptimSetup = greedy_descent_init(initial, ℓ, next_branch, config)

println("Optimization setup complete. Ready to optimize.")

In [ ]:
println("Optimising parameter using Greedy Descent")

println("Loss before starting greedy descent is ", eval_loss(greedy_optim))

# Make 20 steps using greedy descent
N_epochs = 20
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim))
    step!(greedy_optim)
end 
t2 = time()

# Compute the new value of the loss
println("Greedy descent finished in ", t2-t1, " seconds. \nThe Final parameter a is ", greedy_optim.param)

## Comparison with Nonlinear Absolute Polynomials

Now let's compare the performance of the linear representation with a nonlinear absolute polynomial representation of the same problem. We'll create an `AbsolutePolynomialSum` that represents a similar loss function using product terms instead of sums.

In [ ]:
gradient_state = 1
gradient_config = 1

# Reinitialize with the same starting point
initial2 = ValuationPolydisc(fill(K(1), num_parameters), fill(0, num_parameters))

# Now let's optimise using gradient descent 
gradient_optim::OptimSetup = gradient_descent_init(initial2, ℓ, gradient_state, gradient_config)

println("Loss before starting gradient descent is ", eval_loss(gradient_optim))

t3 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(gradient_optim))
    step!(gradient_optim)
end 
t4 = time()

# Compute the new value of the loss
println("Gradient descent finished in ", t4-t3, " Seconds. \nThe Final parameter a is ", gradient_optim.param)

In [ ]:
println("\n=== COMPARISON: Nonlinear Absolute Polynomial ===")
println("Initialising nonlinear model with the same loss function")

# Create the nonlinear absolute polynomial representation
# Using products: |(x_1 - a_1)(x_2 - a_2)...(x_num_data_vars - a_num_data_vars)|
g_nonlinear = AbsolutePolynomialSum([x_vars[i] - a_vars[i] for i in 1:num_data_vars])
f_nonlinear = AbstractModel(g_nonlinear, param_info)

# Create loss function with the same data
ℓ_nonlinear = MPE_loss_init(f_nonlinear, data, 1)

# Initialize with the same starting point
initial_nl = ValuationPolydisc(fill(K(1), num_parameters), fill(0, num_parameters))

# Greedy descent with nonlinear model
greedy_optim_nl::OptimSetup = greedy_descent_init(initial_nl, ℓ_nonlinear, next_branch, config)

println("\n--- Greedy Descent on Nonlinear Model ---")
println("Loss before starting greedy descent is ", eval_loss(greedy_optim_nl))

t5 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_nl))
    step!(greedy_optim_nl)
end 
t6 = time()

println("Greedy descent (nonlinear) finished in ", t6-t5, " seconds. \nThe Final parameter a is ", greedy_optim_nl.param)

In [ ]:
println("\n=== COMPARISON: Nonlinear Absolute Polynomial ===")
println("Initialising nonlinear model with the same loss function")

# Create the nonlinear absolute polynomial representation
g_nonlinear = AbsolutePolynomialSum([x_vars[i] - a_vars[i] for i in 1:20])
f_nonlinear = AbstractModel(g_nonlinear, param_info)

# Create loss function with the same data
ℓ_nonlinear = MPE_loss_init(f_nonlinear, data, 1)

# Initialize with the same starting point
initial_nl = ValuationPolydisc(fill(K(1), 20), fill(0, 20))

# Greedy descent with nonlinear model
greedy_optim_nl::OptimSetup = greedy_descent_init(initial_nl, ℓ_nonlinear, next_branch, config)

println("\n--- Greedy Descent on Nonlinear Model ---")
println("Loss before starting greedy descent is ", eval_loss(greedy_optim_nl))

t5 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_nl))
    step!(greedy_optim_nl)
end 
t6 = time()

println("Greedy descent (nonlinear) finished in ", t6-t5, " seconds. \nThe Final parameter a is ", greedy_optim_nl.param)